# KAI Scheduler

A comprehensive guide to KAI Scheduler for AI/ML workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

KAI Scheduler (KAI-Scheduler) is an **open-source, Kubernetes-native scheduler** from NVIDIA, designed specifically for **AI workloads at large scale**.

### What is it?

- A pluggable Kubernetes scheduler focused on **GPU-accelerated** workloads.  
- Supports **Dynamic Resource Allocation (DRA)**, **GPU sharing**, and **gang scheduling**.  
- Works with standard Kubernetes APIs and ResourceClaims to manage vendor-specific devices (NVIDIA, AMD, etc.).

### Why use it?

Key benefits of using KAI Scheduler:

- **High GPU utilization** via GPU sharing and topology-aware placement.  
- **Gang scheduling** and queueing for multi-pod AI jobs.  
- **Vendor-aware scheduling** that understands GPU topology and capabilities.

### When to use it?

KAI Scheduler is particularly useful when:

- You run **large GPU clusters** on Kubernetes and need to maximize utilization.  
- You want **fine-grained control over GPU allocation** (full, fractional, multiple GPUs per pod).  
- You need **AI-focused scheduling semantics** beyond what the default kube-scheduler offers.

## Key Features

### Core Capabilities of KAI Scheduler

| Feature | Description | Benefit |
|--------|-------------|---------|
| **Dynamic Resource Allocation (DRA)** | Uses Kubernetes ResourceClaims to allocate vendor GPUs and other devices. | Flexible, vendor-aware device management. |
| **GPU sharing** | Allow multiple workloads to share GPUs (e.g., via memory fractions). | Higher GPU utilization, more experiments per GPU. |
| **Gang scheduling** | Schedule related pods together so distributed jobs start coherently. | Better performance and fewer partial-start issues. |
| **Hierarchical queues** | Organize workloads by tenants and priorities. | Multi-tenant governance for AI clusters. |
| **Topology awareness** | Understands GPU/PCIe topology for placement decisions. | Improved performance for communication-heavy models. |

## Architecture Overview

KAI Scheduler plugs into Kubernetes as an alternate scheduler.

```text
+-----------------------------+
|       Kubernetes API        |
+-----------------------------+
          ^          |
          |          | Pod specs with ResourceClaims
          |          v
+----------+------------------+
|       KAI Scheduler         |
|  • Queues & gang scheduling |
|  • GPU/DRA handling         |
+----------+------------------+
           |
           v
+-----------------------------+
|      Kubernetes Nodes       |
|  • GPU devices & drivers    |
+-----------------------------+
```

Pods that should be handled by KAI set their `schedulerName` to `kai-scheduler` and may use ResourceClaims for GPU devices.

## Installation

KAI Scheduler is deployed on a Kubernetes cluster via manifests/Helm from the official GitHub repository.

This is typically handled by **platform or infrastructure teams**, who:

- Install the KAI Scheduler controller and CRDs.  
- Configure GPU device classes and ResourceClaims.  
- Define queues, priorities, and gang scheduling policies.

As an ML engineer, you don’t install KAI yourself; you consume it via pod manifests once it’s in place.

In [ ]:
# KAI Scheduler is installed cluster-wide, not via pip.

print("Work with your platform team to enable KAI Scheduler on your Kubernetes cluster.")

## Basic Usage

Once KAI Scheduler is installed and configured, you typically:

- Add `schedulerName: kai-scheduler` to pods that should be scheduled by KAI.  
- Reference GPU **ResourceClaims** in pod specs to request GPU resources according to platform conventions.

A very simple pod example (conceptual):

In [ ]:
# Example pod using KAI Scheduler (YAML, conceptual)

kai_pod_yaml = """
apiVersion: v1
kind: Pod
metadata:
  name: kai-gpu-job
spec:
  schedulerName: kai-scheduler
  # Example: bind to a pre-created GPU ResourceClaim (details are cluster-specific)
  resourceClaims:
  - name: gpu
    source:
      resourceClaimName: my-gpu-claim
  containers:
  - name: trainer
    image: your-registry/trainer:latest
    resources:
      requests:
        cpu: "4"
        memory: "16Gi"
  restartPolicy: Never
"""

print(kai_pod_yaml)

## Advanced Features

- **GPU sharing**: Request GPU memory fractions or multiple pods per GPU, according to cluster policy.  
- **Gang scheduling**: Submit groups of pods that must start together (for distributed training).  
- **Hierarchical queues and priorities**: Organize workloads by team or project, with specific quotas and priorities.  
- **Topology-aware placement**: Align pods with GPU/PCIe/NVLink topology for performance-sensitive jobs.

In [ ]:
# Placeholder for multi-pod / gang-scheduling examples

print("Refer to the KAI Scheduler docs for examples of gang-scheduled AI workloads.")

## Use Cases

- **Distributed training jobs** that need gang scheduling and GPU topology awareness.  
- **Multi-tenant GPU clusters** where teams share limited GPU resources.  
- **Inference clusters** running many concurrent GPU-backed services with GPU sharing enabled.

## Best Practices

1. **Coordinate closely with platform teams**  
   - Understand available GPU classes, queues, and sharing policies.

2. **Start with non-shared GPU workloads**  
   - Validate correctness before turning on GPU sharing for critical jobs.

3. **Instrument and profile**  
   - Measure performance under different sharing/topology configurations.

4. **Use descriptive labels and annotations**  
   - Help platform teams and dashboards categorize workloads correctly.

## Common Pitfalls

1. **Assuming GPU sharing is free**  
   - Symptom: Performance variability or interference between workloads.  
   - Fix: Test workloads under shared conditions and set SLOs appropriately.

2. **Misconfigured ResourceClaims**  
   - Symptom: Pods pending due to unresolved or invalid claims.  
   - Fix: Validate ResourceClass/ResourceClaim configuration with platform teams.

3. **Unexpected scheduling behavior**  
   - Symptom: Pods not scheduled by KAI when expected.  
   - Fix: Check `schedulerName`, queue configuration, and scheduler logs.

## Performance Optimization

- **Align pod specs with GPU sharing policy** (e.g., memory fractions).  
- **Use topology-aware placement** for communication-heavy training jobs.  
- **Monitor GPU utilization and queue latency** to tune policies over time.

Performance tuning is highly cluster- and workload-specific; rely on benchmarks and observability.

In [ ]:
# Placeholder: performance metrics are exposed via Kubernetes/cluster monitoring

print("Use cluster metrics (GPU utilization, pod wait times) and KAI-specific metrics\n"
      "to guide performance tuning.")

## Production Deployment

- KAI Scheduler should be treated as **core scheduling infrastructure** for AI clusters.  
- Plan for **versioning and upgrades** in coordination with Kubernetes versions and GPU drivers.  
- Define clear **tenant boundaries and queues** to avoid contention and misaligned expectations.

## Monitoring and Observability

- Use **cluster monitoring stacks** (Prometheus/Grafana) for GPU and pod metrics.  
- Leverage **scheduler logs and events** to investigate scheduling decisions.  
- Build dashboards for **queue length, wait times, and GPU usage by tenant**.

## Troubleshooting

- **Pods stuck in `Pending`**:  
  - Verify `schedulerName`, ResourceClaims, and queue configuration.  

- **Unexpected GPU allocation**:  
  - Confirm ResourceClaim parameters and GPU sharing policies.  

- **Scheduler errors**:  
  - Inspect KAI Scheduler logs and verify compatibility with your Kubernetes and GPU stack.

## Comparison with Alternatives

| Aspect | KAI Scheduler | Default kube-scheduler | Run:ai / vendor schedulers |
|--------|--------------|------------------------|----------------------------|
| Focus | AI/GPU workloads | General-purpose | Enterprise GPU platform |
| GPU sharing | Built-in via DRA (cluster-specific) | Limited | Rich, product-specific |
| Open source | Yes | Yes | Typically commercial |

Choose KAI Scheduler when you:

- Want an **open-source, AI-focused scheduler** with GPU sharing and gang scheduling on Kubernetes.

## Resources

- GitHub repository: https://github.com/NVIDIA/KAI-Scheduler  
- Quickstart docs: see the `docs/quickstart` directory in the repo.  
- GPU sharing docs: see the `docs/gpu-sharing` directory.

These contain up-to-date instructions and examples for configuring KAI Scheduler in your cluster.